In [18]:
import pandas as pd

df = pd.read_csv("data/processed/hospital_cleaned.csv")

total_admissions = len(df)
print(f"Total Admissions: {total_admissions:,}")

Total Admissions: 50,213
Average Length of Stay: 4.42 Days


In [19]:
avg_length_of_stay = df["Length_of_Stay_Days"].mean()
print(f"Average Length of Stay: {avg_length_of_stay:.2f} Days")

Average Length of Stay: 4.42 Days


In [20]:
readmission_count = (df["Readmission_Flag"] == "Yes").sum()
readmission_rate = (readmission_count / total_admissions) * 100

print(f"Total Readmissions: {readmission_count:,}")
print(f"Readmission Rate: {readmission_rate:.2f}%")

Total Readmissions: 6,028
Readmission Rate: 12.00%


In [22]:
avg_bed_utilization = df["Bed_Utilization_Rate_Pct"].mean()
print(f"Average Bed Utilization Rate: {avg_bed_utilization:.2f}%")

Average Bed Utilization Rate: 69.96%


In [32]:
df["Occupancy_Rate"] = (
    df["Occupied_Beds_At_Admission"] / df["Total_Beds_In_Department"]
) * 100
avg_occupancy_rate = df["Calculated_Occupancy_Rate"].mean()
print(f"Average Occupancy Rate: {avg_occupancy_rate:.2f}%")

Average Occupancy Rate: 69.96%


In [25]:
avg_dept_efficiency = df["Department_Efficiency_Score"].mean()
print(f"Overall Department Efficiency Score: {avg_dept_efficiency:.2f}")

Overall Department Efficiency Score: 96.88


In [33]:
dept_kpi_summary = (
    df.groupby("Department")
    .agg(
        Total_Admissions=("Patient_ID", "count"),
        Occupancy_Rate=("Occupancy_Rate", lambda x: round(x.mean(), 1)),
        Avg_Length_of_Stay_Days=(
            "Length_of_Stay_Days",
            lambda x: round(x.mean(), 2),
        ),
        Readmission_Rate_Pct=(
            "Readmission_Flag",
            lambda x: round((x == "Yes").mean() * 100, 2),
        ),
        Bed_Utilization_Rate_Pct=(
            "Bed_Utilization_Rate_Pct",
            lambda x: round(x.mean(), 1),
        ),
        Department_Efficiency_Score=(
            "Department_Efficiency_Score",
            lambda x: round(x.mean(), 1),
        ),
    )
    .reset_index()
)
print("\n--- Department-Wise KPI Summary ---")
print(dept_kpi_summary.round(2))


--- Department-Wise KPI Summary ---
         Department  Total_Admissions  Occupancy_Rate  \
0        Cardiology              5010            70.1   
1         Emergency              5063            69.8   
2  General Medicine              5014            69.7   
3        Gynecology              4962            69.9   
4               ICU              4982            70.4   
5         Neurology              5117            69.9   
6          Oncology              5030            70.2   
7       Orthopedics              5046            70.0   
8        Pediatrics              4988            69.9   
9           Surgery              5001            69.8   

   Avg_Length_of_Stay_Days  Readmission_Rate_Pct  Bed_Utilization_Rate_Pct  \
0                     3.95                 12.16                      70.1   
1                     3.99                 12.48                      69.8   
2                     3.90                 11.99                      69.7   
3                     3

In [34]:
output_filename = "hospital_final_dataset.xlsx"
with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Cleaned_Data", index=False)
    dept_kpi_summary.to_excel(
        writer, sheet_name="Department_KPIs", index=False
    )

    # Auto-adjust column widths for Department_KPIs sheet
    worksheet = writer.sheets["Department_KPIs"]
    for col in worksheet.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        col_letter = col[0].column_letter
        worksheet.column_dimensions[col_letter].width = max(max_len + 3, 12)

print("Formatted hospital_final_dataset.xlsx generated successfully!")

Formatted hospital_final_dataset.xlsx generated successfully!
